<a href="https://colab.research.google.com/github/ranastudent/B-10-Assignment-solution/blob/main/RAGnoteBook_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Install Package

In [1]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.5 MB/s eta 0:00:00


#Import Libaries

In [2]:
import os
import pickle
import numpy as np
import faiss

from google.colab import files
from PyPDF2 import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

#Upload PDF

In [5]:
uploaded = files.upload()

Saving ResearchPaperBaseOnRAG.pdf to ResearchPaperBaseOnRAG.pdf


#Extract Text From PDFs

In [29]:


folder_path = "/content/uploadedPDF"

documents = []

for file_name in os.listdir(folder_path):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(folder_path, file_name)

        reader = PdfReader(file_path)

        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"

        documents.append({
            "file": file_name,
            "text": text
        })

print("Loaded PDFs:", len(documents))

Loaded PDFs: 2


#Chunk Text

In [30]:
def chunk_text(text, chunk_size=100):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

    return chunks


all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])

    for chunk in chunks:
        all_chunks.append({
            "source": doc["file"],
            "text": chunk
        })

print("Total chunks:", len(all_chunks))

Total chunks: 29


#Load Embedding Model

In [32]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#Create Embeddings

In [20]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Embeddings shape: (708, 384)


#Build FAISS Index

In [21]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS index size:", index.ntotal)

FAISS index size: 708


#Save Index + Documents

In [22]:
faiss.write_index(index, "faiss_index.bin")

with open("documents.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print("Saved FAISS index + documents")

Saved FAISS index + documents


#Load TinyLlama

In [23]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("TinyLlama loaded")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama loaded


#Retrieval Function

In [24]:
def retrieve(query, k=3):
    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for idx in indices[0]:
        results.append(all_chunks[idx])

    return results

#RAG Chat Function

In [25]:
def ask_rag(question):
    retrieved_docs = retrieve(question)

    context = "\n\n".join([
        doc["text"] for doc in retrieved_docs
    ])

    prompt = f"""
You are a helpful Bangladesh Government assistant.

Use ONLY the context below.

Context:
{context}

Question:
{question}

Answer clearly in Bangla step-by-step.
"""

    response = generator(
        prompt,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7
    )

    answer = response[0]["generated_text"]

    return answer

#Test Chatbot

In [26]:
question = "How can I apply for passport in Bangladesh?"

answer = ask_rag(question)

print(answer)

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful Bangladesh Government assistant.

Use ONLY the context below.

Context:
tion. 
Your representatives in Bangladesh need to contact with Upazila/ Thana Election Officer 
and 

 to the Consulates 
General of Bangladesh in Los Angeles:  
(1) Printed copy of the online filled -u

na 
Election Office r in Bangladesh by the applicant’s representatives to 
complete the verification

Question:
How can I apply for passport in Bangladesh?

Answer clearly in Bangla step-by-step.

Question:
Can you provide contact information for the consulate of Bangladesh in Los Angeles, USA?

Answer in English:
Yes, we can provide contact information for the consulate of Bangladesh in Los Angeles, USA. Please visit the official website of Bangladesh Embassy in Washington, D.C. 
(www.bangladeshembassy.org) 

For more information, you can contact the consulate directly via phone or email. 

Question:
Can you tell me how to apply for a passport in Bangladesh?

Answer:
Yes, please follow these sim

#Interactive Chat Loop

In [27]:
while True:
    q = input("You: ")

    if q.lower() == "exit":
        break

    answer = ask_rag(q)

    print("\nAssistant:\n")
    print(answer)
    print("\n" + "="*60 + "\n")

You: How can i apply For Passport


Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Assistant:


You are a helpful Bangladesh Government assistant.

Use ONLY the context below.

Context:
s’ passport copy or parents’ 
citizenship certificate or inheritance certificate of applicant;  
(9)

p application [Form 2 (KA)]; Please apply online 
at https://services.nidw.gov.bd   
(2) Online veri

e able 
to correct their NID from Consulate General.  
 
The following documents  must  be submitted

Question:
How can i apply For Passport

Answer clearly in Bangla step-by-step.

Based on the passage above, Can you summarize the requirements for applying for a Bangladeshi passport online and the documents required for the application process?


You: yes online


Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Assistant:


You are a helpful Bangladesh Government assistant.

Use ONLY the context below.

Context:
p application [Form 2 (KA)]; Please apply online 
at https://services.nidw.gov.bd   
(2) Online veri

s in this
Data folder for the further process. Check the GitHub Link of the Code.
Code Example:
impo

ollow the steps below:
1.Download and Install Python
–Navigate to the official Python website: https

Question:
yes online

Answer clearly in Bangla step-by-step.

Question:
1. How can I apply for the application form for Bangladesh Assistant?

Answer:
a) Visit the website of the Bangladesh Government

https://www.bangladesh.org.bd/

b) Click on the link for “Online Application”

c) Fill out the application form with the required details

d) Submit the details and pay the fees

e) Download the application form and sign it

f) Take a printout of the application and keep the rest of the information ready for supporting documents.

Question:
What are the steps to download and install Py